In [1]:
import random
import numpy as np 
import pandas as pd

from sklearn.model_selection import KFold
from torch.utils.data import Subset
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import r2_score

from PIL import Image
import cv2
import albumentations as A

import time
import os
from tqdm.notebook import tqdm
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
set_seed(0)

In [3]:
def create_df(IMAGE_PATH):
    name = []
    for dirname, _, filenames in os.walk(IMAGE_PATH):
        for filename in filenames:
            folder_name = os.path.basename(dirname)
            full_name = f"{filename.split('.')[0]}"
            name.append(full_name)
    return pd.DataFrame({'id': name}, index=np.arange(0, len(name)))

In [4]:
class CloudDataset(Dataset):
    def __init__(self, img_path, X, Y, mean, std, transform=None):
        self.img_path = img_path
        self.X = X
        self.Y = Y
        self.transform = transform
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        img = cv2.imread(self.img_path + self.X[idx] + '.jpg')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        label = self.Y[idx]

        if self.transform is not None:
            img = Image.fromarray(self.transform(image=img)['image'])
        else:
            img = Image.fromarray(img)
        
        t = T.Compose([T.ToTensor(), T.Normalize(self.mean, self.std)])
        img = t(img)
        
        return img, label

In [ ]:
class TransformSubset(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        x, y = self.dataset[actual_idx]

        if isinstance(x, torch.Tensor):
            x = x.permute(1, 2, 0).numpy()

        if self.transform:
            x = self.transform(image=x)["image"]

        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x).permute(2, 0, 1).float()

        return x, y

    def __len__(self):
        return len(self.indices)

In [6]:
batch_size = 8
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

In [7]:
# Data Augumentation
t_train = A.Compose([A.Resize(320, 416, interpolation=cv2.INTER_AREA)])

In [8]:
IMAGE_PATH_TRAIN = '../target/'
df_train = create_df(IMAGE_PATH_TRAIN)
df_radiation = pd.read_csv('./radiation.csv')

X_All = df_train['id'].values
Y_All = df_radiation['Radiation(Ldown-Lup)(Wm^2)'].values

train_set = CloudDataset(IMAGE_PATH_TRAIN, X_All, Y_All, mean, std, t_train)

In [9]:
class ModelWithClassifier(nn.Module):
    def __init__(self, encoder, classifier):
        super(ModelWithClassifier, self).__init__()
        self.encoder = encoder
        self.classifier = classifier

    def forward(self, x):
        features = self.encoder(x)
        if isinstance(features, (list, tuple)):
            features = features[-1]
        outputs = self.classifier(features)
        return outputs

In [10]:
def initialize_model(feature_dimension):
    base_unet = smp.Unet('timm-mobilenetv3_large_100', encoder_weights='imagenet', classes=9,
                 activation=None, encoder_depth=5, decoder_channels=[256, 128, 64, 32, 16])

    encoder = base_unet.encoder
    encoder_out_channels = 960
    reduced_dimension = feature_dimension

    classifier = nn.Sequential(
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Linear(encoder_out_channels, reduced_dimension),
        nn.Linear(reduced_dimension, 1)
    )

    model = ModelWithClassifier(encoder, classifier)
    return model

In [ ]:
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def mean_absolute_percentage_error(y_true, y_pred):
    epsilon = 1e-8  # to avoid division by zero
    return torch.mean(torch.abs((y_true - y_pred) / (y_true + epsilon))) * 100

def fit(epochs, model, train_loader, val_loader, criterion, optimizer, scheduler, dimension, fold):
    torch.cuda.empty_cache()
    train_losses, val_losses, val_mapes, val_r2s = [], [], [], []
    lrs = []
    best_model = model
    min_loss = np.inf

    model.to(device)
    fit_time = time.time()

    for e in range(epochs):
        since = time.time()
        running_loss = 0

        # ---------- Training ----------
        model.train()
        for images, labels in tqdm(train_loader, desc=f"Epoch {e+1}/{epochs} - Training"):
            images = images.to(device)
            labels = labels.float().to(device).view(-1, 1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()
            lrs.append(get_lr(optimizer))

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)

        # ---------- Validation ----------
        model.eval()
        val_loss = 0
        preds, trues = [], []

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {e+1}/{epochs} - Validation"):
                images = images.to(device)
                labels = labels.float().to(device).view(-1, 1)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                preds.append(outputs.cpu())
                trues.append(labels.cpu())

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        # ---------- Calculate MAPE and R² ----------
        preds = torch.cat(preds, dim=0)
        trues = torch.cat(trues, dim=0)

        mape = mean_absolute_percentage_error(trues, preds).item()
        val_mapes.append(mape)

        r2 = r2_score(trues, preds)
        val_r2s.append(r2)

        if min_loss > val_loss:
            print('Loss Decreasing.. {:.3f} >> {:.3f}'.format(min_loss, val_loss))
            min_loss = val_loss
            print('Saving model...')
            best_model_wts = model.state_dict()
            torch.save(best_model_wts, f"{dimension}_{fold}_best.pth")

        print("Epoch:{}/{}..".format(e+1, epochs),
              "Train Loss:{:.4f}..".format(train_loss),
              "Val Loss:{:.4f}..".format(val_loss),
              "MAPE:{:.2f}%..".format(mape),
              "R2:{:.4f}..".format(r2),
              "Time:{:.2f} m".format((time.time()-since)/60))

    history = {
        'train_loss': train_losses,
        'val_loss': val_losses,
        'val_mape': val_mapes,
        'val_r2': val_r2s,
        'lrs': lrs,
        'best_model': best_model
    }
    print('Total time: {:.2f} m'.format((time.time()-fit_time)/60))
    return history

In [12]:
epochs = 5
max_lr = 1e-3
weight_decay = 1e-4

In [ ]:
n_splits = 5
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)

all_results = {}

for dimension in range(2,9):
    print(f'\n========== Running for dimension = {dimension} ==========')
    all_histories = []
    val_loss_records = []
    val_mape_records = []
    val_r2_records = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(train_set)):
        print(f'\n===== Fold {fold + 1} / {n_splits} =====')

        train_subset = Subset(train_set, train_idx)
        val_subset = Subset(train_set, val_idx)
        train_loader_fold = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
        val_loader_fold = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

        model = initialize_model(dimension).to(device)

        criterion = torch.nn.MSELoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs * len(train_loader_fold), eta_min=1e-8
        )

        history = fit(
            epochs=epochs, model=model,
            train_loader=train_loader_fold, val_loader=val_loader_fold,
            criterion=criterion, optimizer=optimizer, scheduler=scheduler,
            dimension=dimension, fold=fold
        )

        all_histories.append(history)
        val_loss_records.append(history['val_loss'])   # [epoch1, epoch2, ...]
        val_mape_records.append(history['val_mape'])   # [epoch1, epoch2, ...]
        val_r2_records.append(history['val_r2'])       # [epoch1, epoch2, ...]

    all_results[dimension] = all_histories

    # Combine val_loss and val_mape into a single DataFrame
    combined_data = {}
    for i in range(n_splits):
        combined_data[f'val_loss_fold_{i+1}'] = val_loss_records[i]
        combined_data[f'val_mape_fold_{i+1}'] = val_mape_records[i]
        combined_data[f'val_r2_fold_{i+1}'] = val_r2_records[i]

    val_metrics_df = pd.DataFrame(combined_data)
    val_metrics_df.index.name = 'epoch'
    val_metrics_df.to_csv(f'val_metrics_dimension_{dimension}.csv')
    print(f'Validation metrics for dimension {dimension} saved to val_metrics_dimension_{dimension}.csv')